# PyTorch Logistic Classifier on MNIST

Training a single-layer logistic classifier on MNIST using a custom PyTorch Dataset that reads HDF5 files.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

torch.manual_seed(42)
np.random.seed(42)

# device selection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
class MNISTDataset(Dataset):
    """Custom dataset to load MNIST from HDF5 files."""
    def __init__(self, filepath):
        with h5py.File(filepath, 'r') as f:
            # xdata is already flattened: (N, 784), float32
            self.x = torch.tensor(np.array(f['xdata']), dtype=torch.float32)
            # ydata is one-hot encoded: (N, 10), need to convert to class indices
            y_onehot = np.array(f['ydata'])
            self.y = torch.tensor(np.argmax(y_onehot, axis=1), dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
# find data files - try a few relative paths
def find_file(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Could not find file, tried: {candidates}')

train_path = find_file(['../mnist_traindata.hdf5', 'mnist_traindata.hdf5'])
test_path  = find_file(['../mnist_testdata.hdf5',  'mnist_testdata.hdf5'])

train_dataset = MNISTDataset(train_path)
test_dataset  = MNISTDataset(test_path)

train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=100, shuffle=False)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples:     {len(test_dataset)}')
print(f'Input shape:      {train_dataset[0][0].shape}')

In [ ]:
class LogisticClassifier(nn.Module):
    """Single fully-connected layer (logistic regression) for MNIST."""
    def __init__(self, input_dim=784, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        return self.fc(x)

model = LogisticClassifier().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params}')

In [ ]:
# Using SGD as required, with L2 regularization via weight_decay
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(yb)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += len(yb)
    return running_loss / total, correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_loss += loss.item() * len(yb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return running_loss / total, correct / total

In [ ]:
n_epochs = 50
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(n_epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(model, test_loader, criterion)
    
    train_losses.append(tr_loss)
    test_losses.append(te_loss)
    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | '
              f'train loss: {tr_loss:.4f} | test loss: {te_loss:.4f} | '
              f'train acc: {tr_acc:.4f} | test acc: {te_acc:.4f}')

print(f'\nFinal test accuracy: {test_accs[-1]*100:.2f}%')

In [ ]:
# Learning curves: loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, n_epochs+1), train_losses, label='Train', color='steelblue')
axes[0].plot(range(1, n_epochs+1), test_losses,  label='Test',  color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Learning Curves - Log Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, n_epochs+1), train_accs, label='Train', color='steelblue')
axes[1].plot(range(1, n_epochs+1), test_accs,  label='Test',  color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Learning Curves - Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved: learning_curves.pdf')

In [ ]:
# Confusion matrix on test set
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

cm = confusion_matrix(all_labels, all_preds)
# normalize by true class count
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Normalized Confusion Matrix - MNIST Logistic Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix.pdf', bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.pdf')